# Shared model setup: splits + metrics

One shared definition of train/test split, CV folds, and evaluation
metrics, so 3 different models (built separately) stay comparable.

Each person then builds their own model in the "Your model here" section,
using the same `cv_folds`, `train_df` / `test_df`, and `regression_metrics`.


## 1. Load data

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = r'..\data\model_input\df_model_input.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


(1745282, 80)


,report_date,lm_positive,lm_negative,lm_polarity,uncertainty_ratio,litigious_ratio,constraining_ratio,strong_modal_ratio,weak_modal_ratio,ifo_business_climate,...,log_toas,log_empl,ncliGrowthThisYear,toas_growth,cash_growth,growth_volatility,firm_age,years_in_panel,naics_2digit,is_na_empl
0,2013-12-01,420,652,-0.216418,0.016592,0.018863,0.011103,0.001009,0.004038,100.4,...,7.595447,0.000000,NaN,NaN,NaN,NaN,36.0,0,31,1
1,2014-12-01,265,402,-0.205397,0.020557,0.014647,0.003769,0.001970,0.006167,98.2,...,7.592642,0.000000,-0.034155,-0.002801,0.127094,NaN,37.0,1,31,1
2,2016-12-01,352,677,-0.315841,0.021990,0.012566,0.005463,0.002117,0.005600,101.0,...,7.514275,5.003946,-0.832512,-0.075375,-0.493523,0.564524,39.0,2,31,0
3,2017-12-01,397,547,-0.158898,0.016656,0.015763,0.006186,0.002141,0.004342,104.7,...,7.512180,4.718499,-0.093514,-0.002093,-0.112008,0.444788,40.0,3,31,0
4,2018-12-01,359,513,-0.176606,0.020424,0.012030,0.006295,0.002518,0.004337,101.2,...,7.472122,4.795791,-0.119000,-0.039266,-0.331663,0.419497,41.0,4,31,0


## 2. Config


In [2]:
df['report_year'] = pd.to_datetime(df['report_date']).dt.year
df.drop(columns=['report_date'], inplace=True)
df.head(5)

,lm_positive,lm_negative,lm_polarity,uncertainty_ratio,litigious_ratio,constraining_ratio,strong_modal_ratio,weak_modal_ratio,ifo_business_climate,ifo_business_situation,...,log_empl,ncliGrowthThisYear,toas_growth,cash_growth,growth_volatility,firm_age,years_in_panel,naics_2digit,is_na_empl,report_year
0,420,652,-0.216418,0.016592,0.018863,0.011103,0.001009,0.004038,100.4,99.1,...,0.000000,NaN,NaN,NaN,NaN,36.0,0,31,1,2013
1,265,402,-0.205397,0.020557,0.014647,0.003769,0.001970,0.006167,98.2,98.4,...,0.000000,-0.034155,-0.002801,0.127094,NaN,37.0,1,31,1,2014
2,352,677,-0.315841,0.021990,0.012566,0.005463,0.002117,0.005600,101.0,101.8,...,5.003946,-0.832512,-0.075375,-0.493523,0.564524,39.0,2,31,0,2016
3,397,547,-0.158898,0.016656,0.015763,0.006186,0.002141,0.004342,104.7,107.1,...,4.718499,-0.093514,-0.002093,-0.112008,0.444788,40.0,3,31,0,2017
4,359,513,-0.176606,0.020424,0.012030,0.006295,0.002518,0.004337,101.2,105.3,...,4.795791,-0.119000,-0.039266,-0.331663,0.419497,41.0,4,31,0,2018


In [3]:
YEAR_COL = "report_year"          
TARGET   = "ncliGrowthNextYear"  

EXCLUDE_COLS = ['idnr', 'name', # identifiers
                 'dateinc', # already used in firm age calculation
                 'naics_core_code', # already used in naics_2digit - naics_core_code is a more detailed industry code, but we want to avoid too many dummies. So we use naics_2digit instead. the first 2 digits already represent a broad sector (manufacturing, construction, retail, etc.)
                 'closdate_year' # this is the same year information as report_date
                 ] 

FEATURE_COLS = df.columns.difference([YEAR_COL, TARGET, *EXCLUDE_COLS]).tolist()

CATEGORY_COLS = ['naics_2digit', 'type']  # Categorical columns to be one-hot encoded

MIN_TRAIN_YEARS = 5     # first fold: train on this many years, test on the next
TEST_YEARS = [2020, 2021, 2022, 2023]   

Reasoning of set min_train_year = 5:
- Early years have far fewer, differently-composed firms (panel coverage
expanded over time)
- If a CV fold trains only on those sparse years, its
score reflects a coverage-composition shift, not real forecasting
difficulty.
- `MIN_TRAIN_YEARS = 5` is a starting recommendation: it dilutes the sparse
years to roughly ~15-20% of fold 1's training set 

In [4]:
year_counts = df[YEAR_COL].value_counts().sort_index()
print(year_counts)

years_sorted = sorted(year_counts.index)

# Adjust this to wherever coverage visibly stabilizes in the printout above
SPARSE_CUTOFF_YEAR = 2013

print(f"\n{'min_train_years':>16} {'first_test_year':>16} {'sparse_share':>14}")
for m in range(4, 10):
    if m >= len(years_sorted):
        continue
    train_years = years_sorted[:m]
    total = sum(year_counts[y] for y in train_years)
    sparse = sum(year_counts[y] for y in train_years if y < SPARSE_CUTOFF_YEAR)
    share = sparse / total if total else float("nan")
    first_test_year = years_sorted[m]
    print(f"{m:>16} {first_test_year:>16} {share:>13.1%}")


report_year
2010     13132
2011     12950
2012     21695
2013     39951
2014    138297
2015    145276
2016    149169
2017    190184
2018    201605
2019    201110
2020    219759
2021    225653
2022    184816
2023      1685
Name: count, dtype: int64

 min_train_years  first_test_year   sparse_share
               4             2014         54.5%
               5             2015         21.1%
               6             2016         12.9%
               7             2017          9.2%
               8             2018          6.7%
               9             2019          5.2%


Note: Reasoning about using test years from 2020, explicitly separate the COVID pandemic on the test set only, because:
- This measures "if an unprecedented shock hits, and my model has never seen anything like it, how badly does it break?" That's actually the most realistic deployment scenario - a bank planning financing programs can't guarantee the next crisis looks like something already in its training data. This is the harder, more honest test.

## 3. Fold + metric functions


In [5]:
def rolling_origin_folds(years, min_train_years=4, max_year=None):
    """Expanding-window folds: train on years[:i], test on years[i].

    Example with years 2013..2020, min_train_years=4:
        Fold 1: train <=2016, test 2017
        Fold 2: train <=2017, test 2018
        Fold 3: train <=2018, test 2019
        Fold 4: train <=2019, test 2020
    Only include years in the training set
    """
    years = sorted(set(years))
    return [
        (years[:i], years[i])
        for i in range(min_train_years, len(years))
    ]


def regression_metrics(y_true, y_pred, baseline_value):
    """RMSE, MAE, R2_oos vs. a naive baseline (e.g. train-set mean).

    R2_oos > 0 means the model beats "always predict the baseline value"
    on this data. Use the SAME baseline_value (train-set mean) for every
    model so the comparison is apples-to-apples
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]

    if len(yt) == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "R2_oos": np.nan, "n": 0}

    err = yt - yp
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))
    ss_res = np.sum(err ** 2)
    ss_baseline = np.sum((yt - baseline_value) ** 2)
    r2_oos = float(1 - ss_res / ss_baseline) if ss_baseline > 0 else np.nan

    return {"RMSE": rmse, "MAE": mae, "R2_oos": r2_oos, "n": int(mask.sum())}


## 4. Train/test split + CV folds

- 4.1. Train/test split:
    - Train set: 2012-2019
    - Test set: 2020-2023
    
    Test set is set aside and never touched during tuning
- 4.2. Create multiple folds for cross validation, built from train set only, so a test years can never leak into folds.


In [6]:
# Train/test split
train_df = df[~df[YEAR_COL].isin(TEST_YEARS)]
train_df = train_df[train_df[YEAR_COL] < min(TEST_YEARS)]   # drop anything before test block too, if present
test_df  = df[df[YEAR_COL].isin(TEST_YEARS)]
baseline_value = train_df[TARGET].mean()   # SAME baseline for every model

print(f"train: {train_df.shape}, test {TEST_YEARS}: {test_df.shape}")


train: (1113369, 80), test [2020, 2021, 2022, 2023]: (631913, 80)


In [7]:
min(df[YEAR_COL]), max(df[YEAR_COL]), min(train_df[YEAR_COL]), max(train_df[YEAR_COL]), min(test_df[YEAR_COL]), max(test_df[YEAR_COL])

(2010, 2023, 2010, 2019, 2020, 2023)

In [8]:
years = train_df[YEAR_COL].unique()

cv_folds = rolling_origin_folds(years, min_train_years=MIN_TRAIN_YEARS)

for train_years, test_year in cv_folds:
    print(f"train <= {max(train_years)} ({len(train_years)} yrs)  ->  test {test_year}")


train <= 2014 (5 yrs)  ->  test 2015
train <= 2015 (6 yrs)  ->  test 2016
train <= 2016 (7 yrs)  ->  test 2017
train <= 2017 (8 yrs)  ->  test 2018
train <= 2018 (9 yrs)  ->  test 2019


## 6. Example usage (delete once you've built your own model)

Shows the pattern everyone should follow: use `cv_folds` for tuning,
`train_df`/`test_df` for the final fit, `regression_metrics` for scoring.
Swap in your own model — the surrounding code doesn't change.


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# --- tuning loop shape (adapt to your model's hyperparameters) ---
train_df[FEATURE_COLS] = train_df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
test_df[FEATURE_COLS] = test_df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)

for train_years, test_year in cv_folds:
    fold_train = train_df[train_df[YEAR_COL].isin(train_years)]
    fold_test  = train_df[train_df[YEAR_COL] == test_year]

    numeric_cols = [c for c in FEATURE_COLS if c not in CATEGORY_COLS]

    preprocess = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"), CATEGORY_COLS),
        ("num", SimpleImputer(strategy="median"), numeric_cols),
    ])

    model = Pipeline([
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ])
    model.fit(fold_train[FEATURE_COLS], fold_train[TARGET])
    preds = model.predict(fold_test[FEATURE_COLS])

    m = regression_metrics(fold_test[TARGET].values, preds, fold_train[TARGET].mean())
    print(test_year, m)

# --- final refit + the ONE number to report/compare ---
final_model = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])
final_model.fit(train_df[FEATURE_COLS], train_df[TARGET])
final_preds = final_model.predict(test_df[FEATURE_COLS])
final_metrics = regression_metrics(test_df[TARGET].values, final_preds, baseline_value)
final_metrics


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4716\218136054.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[FEATURE_COLS] = test_df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


2015 {'RMSE': 0.3909876044409199, 'MAE': 0.2898307955675445, 'R2_oos': -0.01703256120474861, 'n': 145276}
2016 {'RMSE': 0.3785901804068197, 'MAE': 0.2782546726052866, 'R2_oos': -0.003273003396582874, 'n': 149169}


MemoryError: Unable to allocate 35.2 MiB for an array with shape (520470, 71) and data type bool

## 7. Your model here

Replace section 6 with your actual model (random forest / XGBoost / NN).
Keep using: `cv_folds` for tuning, `train_df`/`test_df` for the final fit,
`regression_metrics(..., baseline_value)` for scoring. Report your final
`final_metrics` dict back to the team for the comparison table.


In [ ]:
# your model + tuning + final_metrics here


Note when developing the model:
- The growth variable like toas_growth and cash_growth from the feature-engineering notebook are pct_change(), which divides by the previous period's value. If a firm's previous cash or toas was 0, pct_change() returns inf. Should consider how to handle it when developing the model

- 